In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader

In [ ]:
torch.manual_seed(42)

In [ ]:
df = pd.read_csv("/content/fmnist_small.csv")
df

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,9,0,0,0,0,0,0,0,0,0,...,0,7,0,50,205,196,213,165,0,0
1,7,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,1,0,0,0,...,142,142,142,21,0,3,0,0,0,0
3,8,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,8,0,0,0,0,0,0,0,0,0,...,213,203,174,151,188,10,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5995,1,0,0,0,0,0,0,0,0,0,...,69,12,0,0,0,0,0,0,0,0
5996,5,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5997,8,0,0,0,0,0,0,0,0,0,...,39,47,2,0,0,29,0,0,0,0
5998,4,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
#here i have selection for the label and other part
X = df.iloc[:,1:].values
y = df.iloc[:,0].values

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size = 0.2, random_state = 42)

In [ ]:
X_train = X_train/255.0
X_test = X_test/255.0

In [ ]:
#Custom Class
class CustomDataset(Dataset):
  def __init__(self, feature, label):
    self.feature = torch.tensor(feature, dtype = torch.float32)
    self.label = torch.tensor(label, dtype = torch.long)
  def __len__(self):
    return len(self.feature)

  def __getitem__(self,idx):
    return self.feature[idx], self.label[idx]

In [ ]:
train_dataset = CustomDataset(X_train,y_train)
test_dataset = CustomDataset(X_test,y_test)

In [ ]:
train_loader = DataLoader(train_dataset, batch_size = 32, shuffle = True)
test_loader = DataLoader(test_dataset, batch_size = 32, shuffle = False)

In [ ]:
#MODEL Architecture
class MyNN(nn.Module):
  def __init__(self,num_feature):
    super().__init__()
    self.model = nn.Sequential(
        nn.Linear(num_feature,784),
        nn.ReLU(),
        nn.Linear(784,128),
        nn.ReLU(),
        nn.Linear(128,64),
        nn.ReLU(),
        nn.Linear(64,10)

    )
  def forward(self,x):
    return self.model(x)

In [ ]:
epochs = 100
learning_rate = 0.1

In [ ]:
#so here we had call the model  and define the loss and optimizer
model = MyNN(X_train.shape[1])

criterion = nn.CrossEntropyLoss()

optimizer = optim.SGD(model.parameters(),  lr = learning_rate)

In [ ]:
# now here we have to define the loop for the training
for epoch in range(epochs):

  total_epoch_loss = 0
  for batch_feature, batch_label in train_loader:

    #forward
    output = model(batch_feature)

    # calculate the loss
    loss = criterion(output, batch_label)

    #backpropogation
    optimizer.zero_grad()
    loss.backward()

    #update loss
    optimizer.step()
    total_epoch_loss = total_epoch_loss + loss.item()
    avg = total_epoch_loss/len(train_loader)
  print(f"Epoch : {epoch + 1}/{epochs}\tLoss : {avg}")

Epoch : 1/100	Loss : 1.571501563390096
Epoch : 2/100	Loss : 0.8654309423764547
Epoch : 3/100	Loss : 0.6895817242066066
Epoch : 4/100	Loss : 0.6170192416508993
Epoch : 5/100	Loss : 0.5552561650673549
Epoch : 6/100	Loss : 0.5197861747940381
Epoch : 7/100	Loss : 0.47419638047615686
Epoch : 8/100	Loss : 0.453597595791022
Epoch : 9/100	Loss : 0.4251025519768397
Epoch : 10/100	Loss : 0.3997040646274885
Epoch : 11/100	Loss : 0.3791622943679492
Epoch : 12/100	Loss : 0.3750269531706969
Epoch : 13/100	Loss : 0.3445204662034909
Epoch : 14/100	Loss : 0.34046843022108075
Epoch : 15/100	Loss : 0.3080689474443595
Epoch : 16/100	Loss : 0.3151059511800607
Epoch : 17/100	Loss : 0.27685923404991625
Epoch : 18/100	Loss : 0.30035236448049546
Epoch : 19/100	Loss : 0.2669838025420904
Epoch : 20/100	Loss : 0.26428321212530137
Epoch : 21/100	Loss : 0.2401512178281943
Epoch : 22/100	Loss : 0.23698318446675937
Epoch : 23/100	Loss : 0.22482216695944468
Epoch : 24/100	Loss : 0.2240200943996509
Epoch : 25/100	Loss 

In [ ]:
model.eval()

MyNN(
  (model): Sequential(
    (0): Linear(in_features=784, out_features=784, bias=True)
    (1): ReLU()
    (2): Linear(in_features=784, out_features=128, bias=True)
    (3): ReLU()
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): ReLU()
    (6): Linear(in_features=64, out_features=10, bias=True)
  )
)

In [ ]:
#evaluation code

total = 0
correct = 0

with torch.no_grad():
  for batch_feature, batch_label in test_loader:

    output = model(batch_feature)

    _, predicted = torch.max(output, 1)
    total = total + batch_label.shape[0]
    correct = correct + (predicted == batch_label).sum().item()
print(correct/total)


0.8216666666666667
